# Proyecto Oráculo — Lanzador de la Fase 7 (Colab T4)

**Autor:** Juan Felipe Henao · **ADAG1** · Análisis y Diseño de Algoritmos 2026-B

Este notebook ejecuta los experimentos con el modelo real **por pasos que se pueden retomar**. El plan completo (consultas, tiempos, semillas y topes) está en `resultados/fase7_plan.json` del repositorio.

**Cómo usarlo**

1. *Entorno de ejecución ▸ Cambiar tipo de entorno de ejecución ▸ T4 GPU.*
2. Ejecutar la celda 1 y luego *Entorno de ejecución ▸ Reiniciar sesión*.
3. Ejecutar la celda 2 (Drive y variables) y, después, los pasos en orden. Cada paso tiene dos celdas: la primera solo **muestra** cuántas consultas y cuánto tiempo gastaría; la segunda (`--confirmar`) **gasta GPU**, con un tope de consultas nuevas.
4. Si la sesión se cae: volver a ejecutar las celdas 1 y 2 y repetir el paso que estaba corriendo. Lo ya evaluado está en el caché de Drive y no se vuelve a cobrar.

**Orden:** preparar → (a) tiempo → (b) pendiente → (c) estrategia → (d) subespacio → entrega. Si el tiempo de Colab no alcanza, (c) va antes que (d).

**Carpetas en Drive** (`MyDrive/proyecto_oraculo/`): el caché original de la primera entrega (`cache_oraculo.json`) **no se modifica**; la Fase 7 trabaja en `fase7_a4cd966/cache/` (caché recalificado con open-instruct a4cd966 y libro de consultas) y escribe sus resultados en `fase7_a4cd966/resultados/`.

## 1 · Instalación (después, reiniciar la sesión)

In [ ]:
!pip install -q -U "bitsandbytes>=0.46.1" transformers accelerate \
                   nltk spacy emoji langdetect immutabledict pandas matplotlib
!python -m spacy download en_core_web_sm -q

# Verificadores fijados en el commit de referencia del proyecto (CLAUDE.md, regla 13)
![ -d /content/open-instruct ] || git clone -q https://github.com/allenai/open-instruct /content/open-instruct
!git -C /content/open-instruct checkout -q a4cd966a9e202843baba653115045e6fd6f473db || (git -C /content/open-instruct fetch -q origin a4cd966a9e202843baba653115045e6fd6f473db && git -C /content/open-instruct checkout -q a4cd966a9e202843baba653115045e6fd6f473db)
!git -C /content/open-instruct rev-parse HEAD

# Código del proyecto: el fork, instalado como paquete
REPO_FORK = "https://github.com/Pipe-1z/ADA-2026-2-GP01"
![ -d /content/ADA-2026-2-GP01 ] || git clone -q {REPO_FORK} /content/ADA-2026-2-GP01
!git -C /content/ADA-2026-2-GP01 pull -q --ff-only
!git -C /content/ADA-2026-2-GP01 log -1 --format="código del proyecto: %H %s"
!pip install -q -e /content/ADA-2026-2-GP01

import io, urllib.request, zipfile
NLTK_DATA = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages"
for carpeta, nombre in [("tokenizers", "punkt"), ("tokenizers", "punkt_tab"),
                        ("taggers", "averaged_perceptron_tagger"),
                        ("taggers", "averaged_perceptron_tagger_eng")]:
    with urllib.request.urlopen(f"{NLTK_DATA}/{carpeta}/{nombre}.zip") as resp:
        zipfile.ZipFile(io.BytesIO(resp.read())).extractall(f"/root/nltk_data/{carpeta}")

print("LISTO  →  Entorno de ejecución ▸ Reiniciar sesión, y sigue en la celda 2")

## 2 · Drive, variables y comprobación de la GPU (en cada sesión)

In [ ]:
import os, subprocess
from google.colab import drive
drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/proyecto_oraculo"
os.environ["ORACULO_DIR_CACHE"] = f"{BASE}/fase7_a4cd966/cache"
os.environ["ORACULO_DIR_RESULTADOS"] = f"{BASE}/fase7_a4cd966/resultados"
os.environ["ORACULO_OPEN_INSTRUCT"] = "/content/open-instruct"
os.makedirs(os.environ["ORACULO_DIR_CACHE"], exist_ok=True)
os.makedirs(os.environ["ORACULO_DIR_RESULTADOS"], exist_ok=True)
os.chdir("/content/ADA-2026-2-GP01")

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print("GPU:", gpu)
assert "T4" in gpu, "Los experimentos de la Fase 7 se corren en una T4 (CLAUDE.md, regla 14)"
!python scripts/fase7.py estado

## Preparar (sin GPU): caché recalificado con a4cd966 y libro de consultas

In [ ]:
!python scripts/fase7.py preparar --cache-original /content/drive/MyDrive/proyecto_oraculo/cache_oraculo.json

## (a) Costo por consulta y tiempo real contra n (sin caché)

In [ ]:
!python scripts/fase7.py tiempo

In [ ]:
!python scripts/fase7.py tiempo --confirmar

## (b) Evaluación pendiente (0,0,0,6,1): completa el camino del Algoritmo 3 con a4cd966

In [ ]:
!python scripts/fase7.py pendiente

In [ ]:
!python scripts/fase7.py pendiente --confirmar

## (c) Estrategia de mejora: dos ranuras + Successive Halving (n₀ = 2, 2 rondas)

In [ ]:
!python scripts/fase7.py estrategia

In [ ]:
!python scripts/fase7.py estrategia --confirmar

## (d) Subespacio con óptimo exacto (2 opciones por ranura al azar, semilla 7400)

In [ ]:
!python scripts/fase7.py subespacio

In [ ]:
!python scripts/fase7.py subespacio --confirmar

## Entrega: ¿cambia `entrega.json`?

Solo si la mejor configuración de (c) supera a la actual por **al menos 3 puntos** en búsqueda. Si cambia, se valida (87 consultas) y se escribe `entrega_propuesta.json` en los resultados de Drive; después hay que copiarlo al repositorio como `entrega.json`, subirlo al fork y comprobar `notebooks/notebook_evaluacion.ipynb` en Colab.

In [ ]:
!python scripts/fase7.py entrega

In [ ]:
# Solo si la celda anterior dice que cambia. Indicar la semana del curso:
!python scripts/fase7.py entrega --confirmar --semana 11

## Descargar los resultados (para copiarlos a `resultados/` del repositorio y hacer commit)

In [ ]:
import shutil
shutil.make_archive("/content/resultados_fase7", "zip", os.environ["ORACULO_DIR_RESULTADOS"])
from google.colab import files
files.download("/content/resultados_fase7.zip")